In [1]:
from bs4 import BeautifulSoup, Comment
import pandas as pd

import requests
import time
import csv
from io import StringIO

import sys
from pathlib import Path

pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, BASE_URL, HTTP_HEADER
from utils import scrape_per_game_tables

### [Basketball-Reference Bot Policy](https://www.sports-reference.com/bot-traffic.html)

Basketball reference enforces a 1 day block if you are caught accessing the site more than 20 times a second.  All cells that run loops which collect the data should have `time.sleep(x)` in the loop where x $\geq3$.  In what follows we use `time.sleep(3.3)` for good measure.

In [ ]:
scrape_per_game_tables(START_YEAR,END_YEAR)

Scraping data for the 1984-1985 season...
Adding team win/losses...


when we import the datafram from the csv it is important to add a comment keyword or it will mislabel the columns using the descriptive comment instead of the names

In [ ]:
df = pd.read_csv(str(ROOT)+'/data/1980-1981_season/per_game_stats_1980-1981.csv',comment='#')

In [8]:
df.sample(5)

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
329,1984,31.0,51.0,David Pope,22.0,KCK,SF,22.0,0.0,5.9,0.8,2.4,0.321,0.0,0.0,0.0,0.8,2.4,0.327,0.321,0.3,0.6,0.538,0.4,0.4,0.8,0.2,0.1,0.1,0.3,1.4,1.9
19,1984,62.0,20.0,Kareem Abdul-Jabbar,37.0,LAL,C,79.0,79.0,33.3,9.2,15.3,0.599,0.0,0.0,0.0,9.2,15.3,0.600,0.599,3.7,5.0,0.732,2.1,5.8,7.9,3.2,0.8,2.1,2.5,3.0,22.0
12,1984,38.0,44.0,Orlando Woolridge,25.0,CHI,SF,77.0,76.0,36.6,8.8,15.9,0.554,0.0,0.1,0.0,8.8,15.8,0.557,0.554,5.3,6.8,0.785,2.1,3.6,5.6,1.8,0.8,0.5,2.3,2.4,22.9
121,1984,31.0,51.0,Bill Walton,32.0,LAC,C,67.0,37.0,24.6,4.0,7.7,0.521,0.0,0.0,0.0,4.0,7.7,0.523,0.521,2.1,3.0,0.680,2.5,6.4,9.0,2.3,0.7,2.1,2.6,2.7,10.1
156,1984,NaN,NaN,Terry Teagle,24.0,2TM,SG,21.0,3.0,16.6,3.5,6.5,0.540,0.1,0.2,0.5,3.4,6.3,0.541,0.547,1.2,1.7,0.714,1.0,1.0,2.0,0.7,0.6,0.2,0.7,1.7,8.3


Note that players who have been traded during the season show up multiple times.  One for each team as well as an aggregate row where the team is listed as "#TM" and # is the number of teams they have been on. We have made the team wins/losses for these rows NaNs.

In [7]:
df.loc[df['Team']=='2TM']

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
138,1984,NaN,NaN,Paul Thompson,23.0,2TM,SF,49.0,27.0,19.2,3.9,9.4,0.412,0.1,0.6,0.20,3.7,8.8,0.427,0.418,1.4,1.8,0.793,1.2,2.1,3.2,1.6,1.1,0.5,1.2,2.4,9.2
156,1984,NaN,NaN,Terry Teagle,24.0,2TM,SG,21.0,3.0,16.6,3.5,6.5,0.540,0.1,0.2,0.50,3.4,6.3,0.541,0.547,1.2,1.7,0.714,1.0,1.0,2.0,0.7,0.6,0.2,0.7,1.7,8.3
165,1984,NaN,NaN,Edgar Jones,28.0,2TM,PF,44.0,5.0,17.5,3.0,6.3,0.473,0.0,0.1,0.00,3.0,6.2,0.480,0.473,1.9,2.5,0.739,1.1,2.8,3.9,0.7,0.5,0.7,1.4,2.8,7.8
169,1984,NaN,NaN,Fred Roberts,24.0,2TM,PF,74.0,0.0,15.9,2.8,5.6,0.498,0.0,0.0,1.00,2.8,5.6,0.496,0.499,2.0,2.5,0.824,1.1,1.5,2.5,1.2,0.4,0.3,1.2,1.9,7.7
174,1984,NaN,NaN,Ron Brewer,29.0,2TM,SG,20.0,0.0,16.3,3.1,5.9,0.525,0.0,0.1,0.00,3.1,5.8,0.534,0.525,1.2,1.3,0.920,0.5,0.6,1.1,0.9,0.3,0.3,0.5,1.2,7.4
201,1984,NaN,NaN,Marc Iavaroni,28.0,2TM,PF,69.0,43.0,19.3,2.3,5.1,0.458,0.0,0.1,0.00,2.3,5.1,0.463,0.458,1.3,1.9,0.680,1.4,3.0,4.4,1.7,0.5,0.5,1.7,3.1,6.0
205,1984,NaN,NaN,Charles Davis,26.0,2TM,SF,61.0,2.0,12.7,2.5,5.8,0.430,0.0,0.2,0.10,2.5,5.7,0.439,0.431,0.8,1.0,0.823,1.0,1.5,2.5,0.8,0.4,0.1,0.9,1.9,5.9
211,1984,NaN,NaN,Billy Knight,32.0,2TM,SG,68.0,1.0,11.8,2.3,5.2,0.441,0.2,0.4,0.44,2.1,4.8,0.441,0.456,0.9,1.1,0.877,0.7,1.0,1.7,1.2,0.2,0.0,1.0,0.9,5.7
218,1984,NaN,NaN,Charles Jones,27.0,2TM,PF,31.0,4.0,21.5,2.2,4.1,0.528,0.0,0.0,NaN,2.2,4.1,0.528,0.528,1.3,1.9,0.690,2.3,3.6,5.9,0.8,0.7,2.5,0.8,3.5,5.6
230,1984,NaN,NaN,Michael Wilson,25.0,2TM,SG,19.0,0.0,14.1,1.9,4.1,0.468,0.0,0.0,NaN,1.9,4.1,0.468,0.468,1.4,1.9,0.750,0.7,0.9,1.6,1.8,0.7,0.3,1.1,1.1,5.2
